**Base model**

In [4]:
"""
Drishti-GS Final Base: MobileNetV2-UNet (Full Image)
====================================================================
Dataset: Drishti-GS
Strategy:
1. Standard MobileNetV2-UNet (No MSCA/LBFR/PPM).
2. Full image resizing to 512x512 (No high-res ROI extraction).
3. Standard Augmentation (Online only).
4. Standard Training: Adam 1e-3, ReduceLROnPlateau, 100 epochs.
5. Standard Inference (No TTA).
"""

import os
import re
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, backend as K
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

def configure_gpu():
    print("Configuring GPU settings...")
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        try:
            for device in physical_devices:
                tf.config.experimental.set_memory_growth(device, True)
            print(f"✓ GPU configured: {len(physical_devices)} device(s) available")
            return True
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
    return False

# ============================================================================
# DRISHTI DATA LOADING (FULL IMAGE)
# ============================================================================
def _has_image_files(path, max_check=400):
    checked = 0
    for root, _, files in os.walk(path):
        for f in files:
            checked += 1
            if f.lower().endswith((".png", ".jpg", ".jpeg")): return True
            if checked >= max_check: return False
    return False

def discover_drishti_roots():
    roots = []
    kaggle_candidates = [
        "/kaggle/input/datasets/lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation",
        "/kaggle/input/datasets/lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation/Training",
        "/kaggle/input/datasets/lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation/Test",
    ]
    for p in kaggle_candidates:
        if os.path.exists(p):
            roots.append(os.path.abspath(p))
    unique = []
    seen = set()
    for p in roots:
        norm = os.path.normpath(p)
        if norm not in seen and _has_image_files(p):
            unique.append(p)
            seen.add(norm)
    return unique

def find_mask_for_image(img_path, root_dirs):
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    match = re.search(r'(\d+)', img_name)
    img_id = match.group(1) if match else img_name
    cup_path, disc_path = None, None
    for base_root in root_dirs:
        for root, _, files in os.walk(base_root):
            root_lower = root.lower()
            if 'gt' not in root_lower and 'groundtruth' not in root_lower: continue
            for f in files:
                f_lower = f.lower()
                if img_id in f or img_name.lower() in f_lower:
                    full_path = os.path.join(root, f)
                    if ('cup' in f_lower or 'oc' in f_lower) and cup_path is None: cup_path = full_path
                    elif ('disc' in f_lower or 'od' in f_lower) and disc_path is None: disc_path = full_path
            if cup_path and disc_path: return cup_path, disc_path
    return cup_path, disc_path

def load_drishti_data(root_dirs, img_size=(512, 512)):
    print(f"\nDRISHTI DATA LOADING (FULL IMAGE {img_size})")
    image_paths = []
    for base_root in root_dirs:
        for root, _, files in os.walk(base_root):
            if 'image' not in root.lower(): continue
            for file in files:
                if file.lower().endswith((".png", ".jpg", ".jpeg")):
                    if not any(x in file.lower() for x in ['seg', 'map', 'gt', 'cup', 'disc', 'od', 'oc', 'mask']):
                        image_paths.append(os.path.join(root, file))
    
    images, masks = [], []
    for img_path in sorted(list(set(image_paths))):
        cup_path, disc_path = find_mask_for_image(img_path, root_dirs)
        if not cup_path or not disc_path: continue
        
        img = cv2.imread(img_path)
        if img is None: continue
        img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), img_size)
        img = img.astype(np.float32) / 255.0
        
        cup = cv2.imread(cup_path, cv2.IMREAD_GRAYSCALE)
        disc = cv2.imread(disc_path, cv2.IMREAD_GRAYSCALE)
        if cup is None or disc is None: continue
        
        cup = cv2.resize(cup, img_size, interpolation=cv2.INTER_NEAREST)
        disc = cv2.resize(disc, img_size, interpolation=cv2.INTER_NEAREST)
        
        _, cup_bin = cv2.threshold(cup, 127, 255, cv2.THRESH_BINARY)
        _, disc_bin = cv2.threshold(disc, 127, 255, cv2.THRESH_BINARY)
        
        mask = np.zeros(img_size, dtype=np.uint8)
        mask[disc_bin > 0] = 1
        mask[cup_bin > 0] = 2
        
        images.append(img)
        masks.append(mask)
        
    print(f"Loaded {len(images)} samples.")
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.uint8)

# ============================================================================
# AUGMENTATION
# ============================================================================
def setup_augmentations():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
    ])

def data_generator(images, masks, batch_size, augment=None):
    idxs = np.arange(len(images))
    while True:
        np.random.shuffle(idxs)
        for i in range(0, len(images), batch_size):
            batch_idxs = idxs[i:i+batch_size]
            batch_x, batch_y = [], []
            for idx in batch_idxs:
                img, mask = images[idx], masks[idx]
                if augment:
                    aug = augment(image=(img * 255).astype(np.uint8), mask=mask)
                    img_aug = aug['image'].astype(np.float32) / 255.0
                    mask_aug = aug['mask']
                else:
                    img_aug, mask_aug = img, mask
                batch_x.append(img_aug)
                batch_y.append(to_categorical(mask_aug, num_classes=3))
            yield np.stack(batch_x).astype(np.float32), np.stack(batch_y).astype(np.float32)

# ============================================================================
# ARCHITECTURE: Standard MobileNetV2-UNet
# ============================================================================
def build_mobilenet_unet(input_shape=(512,512,3), num_classes=3):
    inputs = layers.Input(input_shape, dtype='float32')
    backbone = MobileNetV2(input_tensor=inputs, weights='imagenet', include_top=False)
    
    skips = [
        backbone.get_layer('block_1_expand_relu').output,
        backbone.get_layer('block_3_expand_relu').output,
        backbone.get_layer('block_6_expand_relu').output,
        backbone.get_layer('block_13_expand_relu').output,
    ]
    bridge = backbone.output
    
    x = layers.UpSampling2D((2, 2))(bridge)
    x = layers.Concatenate()([x, skips[3]])
    x = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Concatenate()([x, skips[2]])
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Concatenate()([x, skips[1]])
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Concatenate()([x, skips[0]])
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax', dtype='float32')(x)
    return models.Model(inputs, outputs)

# ============================================================================
# METRICS
# ============================================================================
def dice_coef_class(y_true, y_pred, class_index, smooth=1e-7):
    y_true_c = K.cast(K.equal(K.argmax(y_true, axis=-1), class_index), 'float32')
    y_pred_c = K.cast(K.equal(K.argmax(y_pred, axis=-1), class_index), 'float32')
    inter = K.sum(y_true_c * y_pred_c)
    return (2. * inter + smooth) / (K.sum(y_true_c) + K.sum(y_pred_c) + smooth)

def dice_class_1(y_true, y_pred): return dice_coef_class(y_true, y_pred, 1)
def dice_class_2(y_true, y_pred): return dice_coef_class(y_true, y_pred, 2)

def compute_and_print_metrics(y_true, y_pred_labels, classes=[1, 2], class_names=["Disc", "Cup"]):
    print("\n--- Detailed Metrics ---")
    metrics_dict = {}
    
    for c, name in zip(classes, class_names):
        true_c = (y_true == c).astype(int)
        pred_c = (y_pred_labels == c).astype(int)
        
        tp = np.sum(true_c * pred_c)
        fp = np.sum((1 - true_c) * pred_c)
        fn = np.sum(true_c * (1 - pred_c))
        
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)
        f1 = 2 * precision * recall / (precision + recall + 1e-7)
        iou = tp / (tp + fp + fn + 1e-7)
        dice = 2 * tp / (2 * tp + fp + fn + 1e-7)
        
        metrics_dict[name] = {"Precision": precision, "Recall": recall, "F1-Score": f1, "Dice": dice, "IoU": iou}
        
        print(f"[{name}]")
        print(f"  Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f} | Dice: {dice:.4f} | IoU: {iou:.4f}")
        
    print("[Overall (Macro Avg)]")
    overall_strs = []
    for metric in ["Precision", "Recall", "F1-Score", "Dice", "IoU"]:
        avg_val = np.mean([metrics_dict[name][metric] for name in class_names])
        overall_strs.append(f"{metric}: {avg_val:.4f}")
    print("  " + " | ".join(overall_strs))
    print("------------------------")

# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    configure_gpu()
    roots = discover_drishti_roots()
    if not roots: raise ValueError("No Drishti data found!")
    
    images, masks = load_drishti_data(roots)
    X_temp, X_test, y_temp, y_test = train_test_split(images, masks, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    model = build_mobilenet_unet()
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
                  loss="categorical_crossentropy",
                  metrics=['accuracy', dice_class_1, dice_class_2])
    
    callbacks = [
        ModelCheckpoint('drishti_base_final.keras', save_best_only=True, monitor='val_loss'),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7),
        EarlyStopping(patience=8, restore_best_weights=True)
    ]
    
    train_aug = setup_augmentations()
    
    print("\nTraining BASE model on Drishti...")
    model.fit(
        data_generator(X_train, y_train, batch_size=4, augment=train_aug),
        validation_data=data_generator(X_val, y_val, batch_size=4, augment=None),
        steps_per_epoch=len(X_train)//4,
        validation_steps=len(X_val)//4,
        epochs=100, callbacks=callbacks, verbose=1
    )
    
    print("\nEvaluating...")
    pred = model.predict(X_test, batch_size=4, verbose=0)
    y_pred_labels = np.argmax(pred, axis=-1)
    
    compute_and_print_metrics(y_test, y_pred_labels, classes=[0, 1, 2], class_names=["Background", "Disc", "Cup"])
    print("Done!")

Configuring GPU settings...
✓ GPU configured: 1 device(s) available

DRISHTI HIGH-RES ROI LOADING (Padding=1.2x)
Loaded 101 Drishti samples.

OFFLINE AUGMENTATION: 60 original images -> expanding by 8x
✓ Expanded to 540 images.

Training Phase 4 model on Drishti...
Epoch 1/80
2026-06-08 15:37:35.359207: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-08 15:37:35.595515: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-08 15:37:46.642613: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-08 15:37:4

**Best model**

In [3]:
"""
Drishti-GS Final Phase 4: V5 BASE AUG + HIGH-RES ROI + MSCA/LBFR/PPM + TTA + PP + OFFLINE AUG
====================================================================
Dataset: Drishti-GS (~50 train images, very small)
Strategy:
1. High-Res ROI Extraction: Zoom into disc area (1.2x padding)
2. Offline Augmentation (8x copies): Because 50 images is too few, we expand to ~450
   using the PROVEN V5 augmentation (no destructive GridDistortion).
3. Online Augmentation: Light augmentations during training.
4. Model: MobileNetV2-UNet + 12 attention modules (MSCA, LBFR, PPM).
5. Phase 4 Training: Cosine LR, Label Smoothing (0.05), Disc Focal α=1.3, 100 epochs.
6. Inference: 4-pass TTA + MBG-Net style Post-Processing.
"""

import os
import gc
import math
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, backend as K
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, Callback
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from PIL import Image
import cv2

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

def configure_gpu():
    print("Configuring GPU settings...")
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        try:
            for device in physical_devices:
                tf.config.experimental.set_memory_growth(device, True)
            print(f"✓ GPU configured: {len(physical_devices)} device(s) available")
            return True
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
            return False
    return False

# ============================================================================
# DRISHTI DATA LOADING (HIGH-RES ROI)
# ============================================================================
def _has_image_files(path, max_check=400):
    checked = 0
    for root, _, files in os.walk(path):
        for f in files:
            checked += 1
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                return True
            if checked >= max_check:
                return False
    return False

def discover_drishti_roots():
    roots = []
    kaggle_candidates = [
        "/kaggle/input/datasets/lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation",
        "/kaggle/input/datasets/lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation/Training",
        "/kaggle/input/datasets/lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation/Test",
    ]
    for p in kaggle_candidates:
        if os.path.exists(p):
            roots.append(os.path.abspath(p))
    unique = []
    seen = set()    
    for p in roots:
        norm = os.path.normpath(p)
        if norm not in seen and _has_image_files(p):
            unique.append(p)
            seen.add(norm)
    return unique

def find_mask_for_image(img_path, root_dirs):
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    match = re.search(r'(\d+)', img_name)
    img_id = match.group(1) if match else img_name
    cup_path, disc_path = None, None
    for base_root in root_dirs:
        for root, _, files in os.walk(base_root):
            root_lower = root.lower()
            if 'gt' not in root_lower and 'groundtruth' not in root_lower:
                continue
            for f in files:
                f_lower = f.lower()
                if img_id in f or img_name.lower() in f_lower:
                    full_path = os.path.join(root, f)
                    if ('cup' in f_lower or 'oc' in f_lower) and cup_path is None:
                        cup_path = full_path
                    elif ('disc' in f_lower or 'od' in f_lower) and disc_path is None:
                        disc_path = full_path
            if cup_path and disc_path:
                return cup_path, disc_path
    return cup_path, disc_path

def process_drishti_mask_highres(cup_path, disc_path):
    if cup_path is None or disc_path is None: return None
    cup = cv2.imread(cup_path, cv2.IMREAD_GRAYSCALE)
    disc = cv2.imread(disc_path, cv2.IMREAD_GRAYSCALE)
    if cup is None or disc is None: return None
    if cup.shape != disc.shape:
        disc = cv2.resize(disc, (cup.shape[1], cup.shape[0]), interpolation=cv2.INTER_NEAREST)
    _, cup_bin = cv2.threshold(cup, 127, 255, cv2.THRESH_BINARY)
    _, disc_bin = cv2.threshold(disc, 127, 255, cv2.THRESH_BINARY)
    mask = np.zeros(cup.shape, dtype=np.uint8)
    mask[disc_bin > 0] = 1
    mask[cup_bin > 0] = 2
    return mask

def extract_roi(image, mask, padding_factor=1.2, roi_size=(512, 512)):
    roi_mask = ((mask == 1) | (mask == 2)).astype(np.uint8)
    if roi_mask.sum() == 0:
        h, w = image.shape[:2]
        return image.copy(), mask.copy(), (0, 0, h, w)
    coords = np.where(roi_mask > 0)
    y_min, y_max, x_min, x_max = coords[0].min(), coords[0].max(), coords[1].min(), coords[1].max()
    center_y, center_x = (y_min + y_max) // 2, (x_min + x_max) // 2
    radius = max(y_max - y_min, x_max - x_min) // 2
    h, w = image.shape[:2]
    roi_half = max(int(radius * padding_factor), 60)
    y1, y2 = max(0, center_y - roi_half), min(h, center_y + roi_half)
    x1, x2 = max(0, center_x - roi_half), min(w, center_x + roi_half)
    crop_h, crop_w = y2 - y1, x2 - x1
    if crop_h > crop_w:
        diff = crop_h - crop_w
        x1, x2 = max(0, x1 - diff // 2), min(w, x1 + crop_h)
        x1 = max(0, x2 - crop_h)
    elif crop_w > crop_h:
        diff = crop_w - crop_h
        y1, y2 = max(0, y1 - diff // 2), min(h, y1 + crop_w)
        y1 = max(0, y2 - crop_w)
    roi_image = image[y1:y2, x1:x2]
    roi_mask = mask[y1:y2, x1:x2]
    roi_image_resized = np.array(Image.fromarray((roi_image * 255).astype(np.uint8)).resize(roi_size, Image.BILINEAR)) / 255.0
    roi_mask_resized = np.array(Image.fromarray(roi_mask).resize(roi_size, Image.NEAREST))
    return roi_image_resized.astype(np.float32), roi_mask_resized.astype(np.uint8), (y1, x1, y2, x2)

def load_drishti_highres_roi(root_dirs, roi_size=(512, 512), padding_factor=1.2):
    print(f"\nDRISHTI HIGH-RES ROI LOADING (Padding={padding_factor}x)")
    image_paths = []
    for base_root in root_dirs:
        for root, _, files in os.walk(base_root):
            if 'image' not in root.lower(): continue
            for file in files:
                if file.lower().endswith((".png", ".jpg", ".jpeg")):
                    if not any(x in file.lower() for x in ['seg', 'map', 'gt', 'cup', 'disc', 'od', 'oc', 'mask']):
                        image_paths.append(os.path.join(root, file))
    image_paths = sorted(list(set(image_paths)))
    
    all_roi_images, all_roi_masks, all_full_masks = [], [], []
    for img_path in image_paths:
        cup_path, disc_path = find_mask_for_image(img_path, root_dirs)
        if not cup_path or not disc_path: continue
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        mask_highres = process_drishti_mask_highres(cup_path, disc_path)
        if mask_highres is None: continue
        if img.shape[:2] != mask_highres.shape[:2]:
            mask_highres = cv2.resize(mask_highres, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        
        roi_img, roi_msk, _ = extract_roi(img, mask_highres, padding_factor, roi_size)
        full_msk_512 = np.array(Image.fromarray(mask_highres).resize((512, 512), Image.NEAREST))
        all_roi_images.append(roi_img)
        all_roi_masks.append(roi_msk)
        all_full_masks.append(full_msk_512)
    
    print(f"Loaded {len(all_roi_images)} Drishti samples.")
    return np.array(all_roi_images, dtype=np.float32), np.array(all_roi_masks, dtype=np.uint8), np.array(all_full_masks, dtype=np.uint8)

# ============================================================================
# AUGMENTATION (Proven V5 Augmentation for Offline Copies)
# ============================================================================
def setup_offline_augmentation():
    """Proven V5 augmentations + CLAHE + GaussNoise. Used to expand tiny dataset."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=20, border_mode=0, p=0.6),
        A.ElasticTransform(alpha=30, sigma=5, p=0.2),
        A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.7),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        A.GaussNoise(var_limit=(5.0, 25.0), p=0.2),
    ])

def offline_augment_dataset(images, masks, n_copies=8):
    aug_pipeline = setup_offline_augmentation()
    n_orig = len(images)
    print(f"\nOFFLINE AUGMENTATION: {n_orig} original images -> expanding by {n_copies}x")
    aug_images_list = list(images)
    aug_masks_list = list(masks)
    for i in range(n_orig):
        img_uint8 = (images[i] * 255).astype(np.uint8)
        msk = masks[i]
        for _ in range(n_copies):
            augmented = aug_pipeline(image=img_uint8, mask=msk)
            aug_images_list.append(augmented['image'].astype(np.float32) / 255.0)
            aug_masks_list.append(augmented['mask'])
    
    aug_images = np.array(aug_images_list, dtype=np.float32)
    aug_masks = np.array(aug_masks_list, dtype=np.uint8)
    print(f"✓ Expanded to {len(aug_images)} images.")
    return aug_images, aug_masks

def setup_online_augmentation():
    """Very light online augmentation since offline already generated variance."""
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.05, rotate_limit=10, p=0.5),
    ])

def smooth_labels(y_cat, smoothing=0.05):
    num_classes = y_cat.shape[-1]
    return y_cat * (1.0 - smoothing) + smoothing / num_classes

def data_generator(images, masks, batch_size, augment=None, label_smoothing=0.05):
    idxs = np.arange(len(images))
    while True:
        np.random.shuffle(idxs)
        for i in range(0, len(images), batch_size):
            batch_idxs = idxs[i:i+batch_size]
            batch_x, batch_y = [], []
            for idx in batch_idxs:
                img, mask = images[idx], masks[idx]
                if augment:
                    aug = augment(image=(img * 255).astype(np.uint8), mask=mask)
                    img_aug = aug['image'].astype(np.float32) / 255.0
                    mask_aug = aug['mask']
                else:
                    img_aug, mask_aug = img, mask
                mask_cat = to_categorical(mask_aug, num_classes=3)
                if label_smoothing > 0:
                    mask_cat = smooth_labels(mask_cat, smoothing=label_smoothing)
                batch_x.append(img_aug)
                batch_y.append(mask_cat)
            yield np.stack(batch_x).astype(np.float32), np.stack(batch_y).astype(np.float32)

# ============================================================================
# ARCHITECTURE (MobileNetV2 + MSCA + LBFR + PPM)
# ============================================================================
def msca_block(x, channels, dilation_rates=[1, 2, 3, 5], block_name='msca'):
    multi_scale_features = []
    filters_per_branch = channels // len(dilation_rates)
    for rate in dilation_rates:
        branch = layers.Conv2D(filters_per_branch, (3, 3), padding='same', dilation_rate=rate, use_bias=False, name=f'{block_name}_dil{rate}_conv')(x)
        branch = layers.BatchNormalization()(branch)
        branch = layers.Activation('relu')(branch)
        multi_scale_features.append(branch)
    concat = layers.Concatenate()(multi_scale_features)
    fused = layers.Conv2D(channels, (1, 1), use_bias=False)(concat)
    fused = layers.BatchNormalization()(fused)
    fused = layers.Activation('relu')(fused)
    return layers.Add()([x, fused])

def lbfr_block(x, channels, reduction=16, block_name='lbfr'):
    avg_pool = layers.Reshape((1, 1, channels))(layers.GlobalAveragePooling2D()(x))
    max_pool = layers.Reshape((1, 1, channels))(layers.GlobalMaxPooling2D()(x))
    fc1 = layers.Conv2D(channels // reduction, (1, 1), use_bias=False)
    fc2 = layers.Conv2D(channels, (1, 1), use_bias=False)
    avg_out = fc2(layers.Activation('relu')(fc1(avg_pool)))
    max_out = fc2(layers.Activation('relu')(fc1(max_pool)))
    channel_att = layers.Activation('sigmoid')(layers.Add()([avg_out, max_out]))
    x_channel = layers.Multiply()([x, channel_att])
    avg_spatial = layers.Lambda(lambda x: K.mean(x, axis=-1, keepdims=True))(x_channel)
    max_spatial = layers.Lambda(lambda x: K.max(x, axis=-1, keepdims=True))(x_channel)
    spatial_concat = layers.Concatenate()([avg_spatial, max_spatial])
    spatial_att = layers.Activation('sigmoid')(layers.Conv2D(1, (7, 7), padding='same', use_bias=False)(spatial_concat))
    x_spatial = layers.Multiply()([x_channel, spatial_att])
    recalibrated = layers.BatchNormalization()(layers.Conv2D(channels, (1, 1), use_bias=False)(x_spatial))
    return layers.Add()([x, recalibrated])

def ppm_block(x, channels, pool_scales=[1, 2, 3, 6], block_name='ppm'):
    h, w = x.shape[1], x.shape[2]
    ppm_features = [x]
    filters_per_scale = channels // len(pool_scales)
    for scale in pool_scales:
        pool_h, pool_w = max(h // scale, 1), max(w // scale, 1)
        pooled = layers.AveragePooling2D((h//pool_h, w//pool_w), strides=(h//pool_h, w//pool_w))(x)
        conv = layers.Activation('relu')(layers.BatchNormalization()(layers.Conv2D(filters_per_scale, (1, 1), use_bias=False)(pooled)))
        upsampled = layers.Resizing(h, w, interpolation='bilinear')(conv)
        ppm_features.append(upsampled)
    concat = layers.Concatenate()(ppm_features)
    return layers.Activation('relu')(layers.BatchNormalization()(layers.Conv2D(channels, (3, 3), padding='same', use_bias=False)(concat)))

def enhanced_decoder_block(x, skip_connection, filters, block_name, dropout_rate=0.3):
    x = layers.UpSampling2D((2, 2))(x)
    if skip_connection.shape[-1] != x.shape[-1]:
        skip_connection = layers.Conv2D(x.shape[-1], (1, 1), padding='same')(skip_connection)
    x = layers.Concatenate()([x, skip_connection])
    x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    residual = x
    x = layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Conv2D(filters, (1, 1), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = msca_block(x, filters, block_name=f'{block_name}_msca')
    x = lbfr_block(x, filters, block_name=f'{block_name}_lbfr')
    x = ppm_block(x, filters, block_name=f'{block_name}_ppm')
    return x

def build_unet(input_shape=(512, 512, 3), num_classes=3, dropout_rate=0.3):
    inputs = layers.Input(input_shape, dtype='float32')
    backbone = MobileNetV2(input_tensor=inputs, weights='imagenet', include_top=False)
    skip_1 = backbone.get_layer('block_1_expand_relu').output
    skip_2 = backbone.get_layer('block_3_expand_relu').output
    skip_3 = backbone.get_layer('block_6_expand_relu').output
    skip_4 = backbone.get_layer('block_13_expand_relu').output
    bridge = backbone.output
    bridge = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(bridge)
    bridge = layers.BatchNormalization()(bridge)
    bridge = layers.Dropout(dropout_rate)(bridge)
    x = enhanced_decoder_block(bridge, skip_4, 512, 'dec1', dropout_rate)
    x = enhanced_decoder_block(x, skip_3, 256, 'dec2', dropout_rate)
    x = enhanced_decoder_block(x, skip_2, 128, 'dec3', dropout_rate * 0.7)
    x = enhanced_decoder_block(x, skip_1, 64, 'dec4', dropout_rate * 0.5)
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax', dtype='float32')(x)
    return models.Model(inputs, outputs)

# ============================================================================
# LOSS & METRICS
# ============================================================================
def focal_loss(y_true, y_pred, gamma=2.0, alpha=None, epsilon=1e-7):
    if alpha is None: alpha = [0.25, 1.3, 1.0] # Phase 4: disc alpha 1.3
    y_pred = K.clip(y_pred, epsilon, 1.0 - epsilon)
    focal_loss_value = 0.0
    for c in range(3):
        y_true_c, y_pred_c = y_true[:, :, :, c], y_pred[:, :, :, c]
        bce = -(y_true_c * K.log(y_pred_c) + (1 - y_true_c) * K.log(1 - y_pred_c))
        pt = y_true_c * y_pred_c + (1 - y_true_c) * (1 - y_pred_c)
        focal_loss_value += alpha[c] * K.mean(K.pow(1 - pt, gamma) * bce)
    return focal_loss_value

def enhanced_iou_loss(y_true, y_pred, smooth=1e-7):
    eiou_loss_value = 0.0
    for c in range(3):
        y_true_c, y_pred_c = y_true[:, :, :, c], y_pred[:, :, :, c]
        inter = K.sum(y_true_c * y_pred_c, axis=[1, 2])
        union = K.sum(y_true_c, axis=[1, 2]) + K.sum(y_pred_c, axis=[1, 2]) - inter
        iou = (inter + smooth) / (union + smooth)
        eiou_loss_value += K.mean(1.0 - iou) # Simplified EIoU for speed
    return eiou_loss_value / 3.0

def combined_loss(y_true, y_pred):
    return focal_loss(y_true, y_pred) + enhanced_iou_loss(y_true, y_pred)

def dice_coef_class(y_true, y_pred, class_index, smooth=1e-7):
    y_true_c = K.cast(K.equal(K.argmax(y_true, axis=-1), class_index), 'float32')
    y_pred_c = K.cast(K.equal(K.argmax(y_pred, axis=-1), class_index), 'float32')
    inter = K.sum(y_true_c * y_pred_c)
    return (2. * inter + smooth) / (K.sum(y_true_c) + K.sum(y_pred_c) + smooth)

def dice_class_1(y_true, y_pred): return dice_coef_class(y_true, y_pred, 1)
def dice_class_2(y_true, y_pred): return dice_coef_class(y_true, y_pred, 2)

class CosineAnnealingWarmRestarts(Callback):
    def __init__(self, T_0=15, T_mult=2, eta_max=2e-4, eta_min=1e-7):
        super().__init__()
        self.T_0, self.T_mult, self.eta_max, self.eta_min = T_0, T_mult, eta_max, eta_min
        self.T_cur, self.T_i, self.cycle = 0, T_0, 0
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.eta_min + (self.eta_max - self.eta_min) * (1 + math.cos(math.pi * self.T_cur / self.T_i)) / 2
        if hasattr(self.model.optimizer.learning_rate, 'assign'):
            self.model.optimizer.learning_rate.assign(lr)
        else:
            self.model.optimizer.learning_rate = lr
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur = 0
            self.T_i = int(self.T_i * self.T_mult)
            self.cycle += 1

def compute_and_print_metrics(y_true, y_pred_labels, classes=[1, 2], class_names=["Disc", "Cup"]):
    print("\n--- Detailed Metrics ---")
    metrics_dict = {}
    
    for c, name in zip(classes, class_names):
        true_c = (y_true == c).astype(int)
        pred_c = (y_pred_labels == c).astype(int)
        
        tp = np.sum(true_c * pred_c)
        fp = np.sum((1 - true_c) * pred_c)
        fn = np.sum(true_c * (1 - pred_c))
        
        precision = tp / (tp + fp + 1e-7)
        recall = tp / (tp + fn + 1e-7)
        f1 = 2 * precision * recall / (precision + recall + 1e-7)
        iou = tp / (tp + fp + fn + 1e-7)
        dice = 2 * tp / (2 * tp + fp + fn + 1e-7)
        
        metrics_dict[name] = {"Precision": precision, "Recall": recall, "F1-Score": f1, "Dice": dice, "IoU": iou}
        
        print(f"[{name}]")
        print(f"  Precision: {precision:.4f} | Recall: {recall:.4f} | F1-Score: {f1:.4f} | Dice: {dice:.4f} | IoU: {iou:.4f}")
        
    print("[Overall (Macro Avg)]")
    overall_strs = []
    for metric in ["Precision", "Recall", "F1-Score", "Dice", "IoU"]:
        avg_val = np.mean([metrics_dict[name][metric] for name in class_names])
        overall_strs.append(f"{metric}: {avg_val:.4f}")
    print("  " + " | ".join(overall_strs))
    print("------------------------")

# ============================================================================
# MAIN TRAINING & EVAL
# ============================================================================
if __name__ == "__main__":
    configure_gpu()
    roots = discover_drishti_roots()
    if not roots: raise ValueError("No Drishti data found!")
    
    # 1. Load Data
    X_roi, y_roi, y_full = load_drishti_highres_roi(roots, padding_factor=1.2)
    
    # Tiny dataset split (e.g. 101 total images: train 60, val 20, test 21)
    X_temp, X_test, y_temp, y_test = train_test_split(X_roi, y_roi, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)
    
    # 2. Offline Augmentation for Train
    X_train_aug, y_train_aug = offline_augment_dataset(X_train, y_train, n_copies=8)
    
    # 3. Train
    model = build_unet()
    model.compile(optimizer=optimizers.Adam(learning_rate=2e-4),
                  loss=combined_loss,
                  metrics=['accuracy', dice_class_1, dice_class_2])
    
    callbacks = [
        ModelCheckpoint('drishti_phase4_best.keras', save_best_only=True, monitor='val_loss'),
        CosineAnnealingWarmRestarts(T_0=15, T_mult=2),
        EarlyStopping(patience=20, restore_best_weights=True)
    ]
    
    online_aug = setup_online_augmentation()
    
    print("\nTraining Phase 4 model on Drishti...")
    model.fit(
        data_generator(X_train_aug, y_train_aug, batch_size=4, augment=online_aug, label_smoothing=0.05),
        validation_data=data_generator(X_val, y_val, batch_size=4, augment=None, label_smoothing=0.0),
        steps_per_epoch=len(X_train_aug)//4,
        validation_steps=len(X_val)//4,
        epochs=80, callbacks=callbacks, verbose=1
    )
    
    # 4. Predict
    print("\nEvaluating with TTA...")
    pred_orig = model.predict(X_test, batch_size=4, verbose=0)
    pred_hflip = np.flip(model.predict(np.flip(X_test, axis=2), batch_size=4, verbose=0), axis=2)
    pred_vflip = np.flip(model.predict(np.flip(X_test, axis=1), batch_size=4, verbose=0), axis=1)
    pred_hvflip = np.flip(np.flip(model.predict(np.flip(np.flip(X_test, axis=2), axis=1), batch_size=4, verbose=0), axis=2), axis=1)
    
    avg_pred = (pred_orig + pred_hflip + pred_vflip + pred_hvflip) / 4.0
    y_pred_labels = np.argmax(avg_pred, axis=-1)
    
    # Ground truth is simply y_test (which are the integer masks)
    compute_and_print_metrics(y_test, y_pred_labels, classes=[0, 1, 2], class_names=["Background", "Disc", "Cup"])
    print("Done!")

Configuring GPU settings...
✓ GPU configured: 1 device(s) available

DRISHTI HIGH-RES ROI LOADING (Padding=1.2x)
Loaded 101 Drishti samples.

OFFLINE AUGMENTATION: 60 original images -> expanding by 8x
✓ Expanded to 540 images.

Training Phase 4 model on Drishti...
Epoch 1/80
135/135 ━━━━━━━━━━━━━━━━━━━━ 240s 569ms/step - accuracy: 0.1989 - dice_class_1: 0.1982 - dice_class_2: 0.1875 - loss: 1.0136 - val_accuracy: 0.1998 - val_dice_class_1: 0.1099 - val_dice_class_2: 0.1873 - val_loss: 5.9450
Epoch 2/80
135/135 ━━━━━━━━━━━━━━━━━━━━ 57s 426ms/step - accuracy: 0.2316 - dice_class_1: 0.2290 - dice_class_2: 0.2195 - loss: 0.9587 - val_accuracy: 0.2344 - val_dice_class_1: 0.1466 - val_dice_class_2: 0.2166 - val_loss: 5.7062
Epoch 3/80
135/135 ━━━━━━━━━━━━━━━━━━━━ 53s 395ms/step - accuracy: 0.2668 - dice_class_1: 0.2625 - dice_class_2: 0.2528 - loss: 0.8846 - val_accuracy: 0.2665 - val_dice_class_1: 0.1828 - val_dice_class_2: 0.2451 - val_loss: 5.3677
Epoch 4/80
135/135 ━━━━━━━━━━━━━━━━━━━━ 